# Gold dimension: `dim_flood_susceptibility`

## Purpose

Publish the MGB flood-susceptibility levels that the project fact and the regional exposure fact join to.

## Grain

One susceptibility classification per source version, plus key `0`.

- Uniqueness: `source_system`, `flood_susceptibility_level`, `source_version`.
- Key `0` is `Unknown`. Facts use it when Silver did not publish one approved level. That covers ambiguous points, unmatched points, and projects without coordinates.

`Unknown` exists only in Gold. Silver never maps a code to `Unknown`, and key `0` never means low or zero risk.

## Safe reruns

`CREATE TABLE IF NOT EXISTS` and `MERGE` on the deterministic key. Rows are never deleted.

## 1. Confirm upstream safety and read the approved mapping

**What:** check the latest Silver CONFIG validation. Then read only active, `APPROVED` rows from `config_mgb_susceptibility_mapping`.

**Why:** levels and ranks come only from governed configuration. This notebook contains no code-to-level mapping of its own.

**Protects:** exactly one mapping version and one source version must be active. Levels and ranks must be one-to-one. Rank `0` is reserved for `Unknown`.

In [ ]:
%sql
USE CATALOG `buildabida-capstone`;
CREATE SCHEMA IF NOT EXISTS `03-gold`;

DECLARE OR REPLACE VARIABLE mgb_mapping_source_system STRING DEFAULT 'DENR MGB flood susceptibility';
DECLARE OR REPLACE VARIABLE flood_rule_version STRING DEFAULT 'gold-dim-flood-susceptibility-v1';

CREATE OR REPLACE TEMPORARY VIEW latest_config_validation AS
SELECT * EXCEPT (validation_rank)
FROM (
    SELECT *, DENSE_RANK() OVER (ORDER BY run_ts DESC, run_id DESC) AS validation_rank
    FROM `04-validation`.silver_config_dq_results
    WHERE layer = 'silver_config'
)
WHERE validation_rank = 1;

SELECT ASSERT_TRUE(
    COUNT(*) > 0 AND COUNT_IF(action = 'stop' AND status IN ('FAIL', 'ERROR')) = 0,
    'STOP: latest Silver configuration validation is missing or unsafe'
)
FROM latest_config_validation;

CREATE OR REPLACE TEMPORARY VIEW approved_mgb_levels AS
SELECT
    mapping.standardized_level AS flood_susceptibility_level,
    mapping.severity_rank,
    mapping.source_system,
    mapping.source_version,
    mapping.mapping_version
FROM `02-silver`.config_mgb_susceptibility_mapping AS mapping
WHERE mapping.source_system = mgb_mapping_source_system
  AND mapping.approval_status = 'APPROVED'
  AND mapping.is_active = TRUE;

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT(DISTINCT mapping_version) = 1
    AND COUNT(DISTINCT source_version) = 1,
    'STOP: exactly one active approved MGB mapping version and source version are required'
)
FROM approved_mgb_levels;

SELECT ASSERT_TRUE(
    COUNT(DISTINCT flood_susceptibility_level) = COUNT(DISTINCT severity_rank)
    AND COUNT(DISTINCT flood_susceptibility_level) = COUNT(DISTINCT NAMED_STRUCT(
        'level', flood_susceptibility_level, 'rank', severity_rank
    ))
    AND COUNT_IF(severity_rank IS NULL OR severity_rank <= 0) = 0
    AND COUNT_IF(flood_susceptibility_level IS NULL OR UPPER(flood_susceptibility_level) = 'UNKNOWN') = 0,
    'STOP: approved MGB levels and ranks must be one-to-one, positive, and not Unknown'
)
FROM approved_mgb_levels;

CREATE OR REPLACE TEMPORARY VIEW latest_mgb_load AS
SELECT * EXCEPT (terminal_rank)
FROM (
    SELECT *, ROW_NUMBER() OVER (
        PARTITION BY table_name
        ORDER BY COALESCE(completed_at, started_at) DESC, started_at DESC
    ) AS terminal_rank
    FROM `01-bronze`.load_log
    WHERE table_name = 'flood_susceptibility'
      AND status IN ('STARTED', 'SUCCESS', 'FAILED', 'SKIPPED_IDEMPOTENT')
)
WHERE terminal_rank = 1;

SELECT ASSERT_TRUE(
    COUNT(*) = 1 AND COUNT_IF(status IN ('SUCCESS', 'SKIPPED_IDEMPOTENT')) = 1,
    'STOP: latest MGB Bronze load is not safe'
)
FROM latest_mgb_load;

## 2. Create the table

**What:** create `03-gold.dim_flood_susceptibility` with the column names, types, and order in `docs/gold_model.md`.

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS `03-gold`.dim_flood_susceptibility (
    flood_susceptibility_key INT NOT NULL COMMENT 'Surrogate key. HASH of the source system, level, and source version. Key 0 is Unknown.',
    flood_susceptibility_level STRING COMMENT 'Approved MGB level: Low, Moderate, High, or Very High. Unknown for key 0.',
    severity_rank INT COMMENT 'Approved severity rank from 1 (Low) to 4 (Very High). 0 for Unknown. Use it to sort, not to add.',
    source_system STRING COMMENT 'Source system of the susceptibility codes.',
    source_version STRING COMMENT 'Source contract of the approved code-to-level mapping.',
    run_id STRING COMMENT 'Deterministic Gold run ID. Hash of the mapping version, source version, and Gold rule version.',
    source_load_ts TIMESTAMP COMMENT 'Bronze load time of the MGB snapshot these levels classify.'
)
USING DELTA
COMMENT 'Gold flood-susceptibility dimension. Approved MGB levels plus key 0 Unknown. Unknown means no single approved level was published. It never means low or zero risk.';

## 3. Build and merge the levels

**What:** build one row per approved level, add key `0`, and merge by key.

**Why:** the key is `HASH` of the source system, level, and source version, so it is the same on every run. A new mapping source version produces new keys and keeps the old rows.

**Expected validation:** keys are unique, no approved level gets key `0`, and ranks are unique.

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW dim_flood_susceptibility_source AS
WITH selected AS (
    SELECT
        MAX(levels.mapping_version) AS mapping_version,
        MAX(levels.source_version) AS source_version,
        MAX(load.completed_at) AS mgb_source_load_ts
    FROM approved_mgb_levels AS levels
    CROSS JOIN latest_mgb_load AS load
),
distinct_levels AS (
    SELECT DISTINCT
        flood_susceptibility_level,
        severity_rank,
        source_system,
        source_version
    FROM approved_mgb_levels
),
approved AS (
    SELECT
        HASH(CONCAT_WS(
            '|', levels.source_system, levels.flood_susceptibility_level, levels.source_version
        )) AS flood_susceptibility_key,
        levels.flood_susceptibility_level,
        levels.severity_rank,
        levels.source_system,
        levels.source_version
    FROM distinct_levels AS levels
),
unknown_level AS (
    SELECT
        0 AS flood_susceptibility_key,
        'Unknown' AS flood_susceptibility_level,
        0 AS severity_rank,
        mgb_mapping_source_system AS source_system,
        selected.source_version
    FROM selected
),
all_levels AS (
    SELECT * FROM unknown_level
    UNION ALL
    SELECT * FROM approved
)
SELECT
    all_levels.*,
    SHA2(CONCAT_WS(
        '|', selected.mapping_version, selected.source_version, flood_rule_version
    ), 256) AS run_id,
    selected.mgb_source_load_ts AS source_load_ts
FROM all_levels
CROSS JOIN selected;

SELECT ASSERT_TRUE(
    COUNT(*) = COUNT(DISTINCT flood_susceptibility_key)
    AND COUNT(*) = COUNT(DISTINCT severity_rank)
    AND COUNT_IF(flood_susceptibility_key = 0) = 1
    AND COUNT_IF(flood_susceptibility_key = 0 AND flood_susceptibility_level <> 'Unknown') = 0,
    'STOP: dim_flood_susceptibility keys or ranks are not unique, or a level collides with key 0'
)
FROM dim_flood_susceptibility_source;

MERGE INTO `03-gold`.dim_flood_susceptibility AS target
USING dim_flood_susceptibility_source AS source
ON target.flood_susceptibility_key = source.flood_susceptibility_key
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

# Summary

Created or refreshed `03-gold.dim_flood_susceptibility` with the approved MGB levels and key `0` `Unknown`. Levels and ranks come only from approved configuration. Next: `03_gold_dim_project_status`.

In [ ]:
%sql
SELECT
    flood_susceptibility_key,
    flood_susceptibility_level,
    severity_rank,
    source_version
FROM `03-gold`.dim_flood_susceptibility
ORDER BY severity_rank;